# Python for MATH 21100 Labs (Optional, Self-Paced)

This notebook is optional and ungraded. It assumes you have programmed before in
some language and brings you up to the minimal Python the labs use: numpy arrays,
writing mathematical functions, solving a linear system, and plotting. It is not a
first course in programming, so it spends no time on what a variable or a loop is,
only on how Python and numpy express them and where the common traps are.

Coding is never graded in this course. If you prefer, skip to Lab A and use an AI
assistant for the code. The value of this tutorial is that afterwards you can read,
check, and modify the code a lab gives you, and recognise when an answer is wrong.

Work top to bottom and run each cell with Shift+Enter, or use Runtime > Run all.

## 0. How Colab runs a notebook

A notebook is an ordered list of cells. A code cell runs its Python when you press
Shift+Enter, and its output (anything printed, plus the last expression's value and
any plot) appears directly beneath it.

All cells share a single memory. A name defined in one cell is visible in later
cells, but only after you have actually run the earlier cell, and in the order you
ran them, not the order they appear on screen. This is the most common source of
confusion: editing a cell and re-running only that cell can leave stale values
elsewhere. When in doubt, Runtime > Restart session and run all gives a clean,
top-to-bottom execution.

## 1. Python recap

Three things whose Python syntax you will need: functions, f-strings for readable
output, and the fact that indentation (not braces) marks a block.

In [ ]:
# Python is dynamically typed: you do not declare types, they are inferred.
n = 5                      # an int
name = "Chebyshev"         # a str
ratio = 3 / 2             # division with / always gives a float: 1.5, not 1

# An f-string (the f before the quote) drops variable values into text.
# The :.3f format shows 3 digits after the decimal point.
print(f"n = {n}, name = {name}, ratio = {ratio:.3f}")

# A function. The body is indented; the docstring in triple quotes documents it.
def square(x):
    """Return x times x."""
    return x * x

print("square(7) =", square(7))

# A for-loop over range(1,5) = 1,2,3,4 (the upper end is excluded), with an if.
total = 0
for k in range(1, 5):
    if k % 2 == 0:         # % is remainder; k is even when k % 2 == 0
        total += k         # += adds in place
print("sum of evens in 1..4 =", total)

## 2. numpy arrays

Numerical code uses numpy arrays, not Python lists. An array holds numbers of one
type (here floats), knows its shape, and supports arithmetic that applies to every
entry at once. That elementwise arithmetic is the single most important habit to
absorb.

In [ ]:
import numpy as np       # np is the universal abbreviation

v = np.array([1.0, 2.0, 3.0, 4.0])
print("v         =", v)
print("shape      =", v.shape)   # (4,) means a 1D array of length 4
print("dtype      =", v.dtype)   # float64, the default floating type

# Arithmetic is ELEMENTWISE. There is no loop and no need for one.
print("v + 10    =", v + 10)     # adds 10 to each entry
print("v * v     =", v * v)      # squares each entry (NOT a dot product)

# Indexing and slicing. Indices start at 0; negatives count from the end.
print("v[0], v[-1] =", v[0], v[-1])   # first and last
print("v[1:3]    =", v[1:3])          # entries 1 and 2; the end index 3 is excluded

**Making grids.** Two functions build the number grids the labs need. `linspace`
gives a fixed count of evenly spaced points including both endpoints; `arange` gives
points at a fixed step with the end excluded. Note the off-by-one: `n+1` points span
`n` equal gaps, which is why the labs ask for `n+1` nodes to get a degree-`n`
polynomial.

In [ ]:
print("linspace(0,1,5)  ->", np.linspace(0, 1, 5))   # 5 points, includes 0 and 1
print("arange(0,5)      ->", np.arange(0, 5))        # 0,1,2,3,4  (5 excluded)
print("arange(0,1,0.25) ->", np.arange(0, 1, 0.25))  # step of 0.25, 1 excluded

## 3. Mathematical functions on arrays

Write a function using numpy operations and it works on a scalar or a whole array
with no change. This is called vectorization, and it is how every lab evaluates a
function on a grid. Applying the array to `np.abs`, `np.max`, `np.sum`, and similar
reductions collapses it to a single number.

In [ ]:
def f(x):
    """Runge's function, vectorized: x may be a scalar or a numpy array."""
    return 1.0 / (1.0 + 25.0 * x**2)     # every operation here is elementwise

grid = np.linspace(-1, 1, 9)             # 9 sample points
vals = f(grid)                           # 9 values, computed at once
print("values       :", np.round(vals, 3))
print("largest value:", np.max(vals))
print("largest |val|:", np.max(np.abs(vals)))   # abs first, then max
print("mean value   :", np.mean(vals))

**Why this matters for the labs.** The error of an approximation is measured as
`np.max(np.abs(f(grid) - p(grid)))`, the largest gap between the true function and
the approximation over a fine grid. That one line is the workhorse of Lab A.

## 4. Loops versus vectorization

The vectorized form is shorter, faster, and less error-prone, so prefer it. A loop
is the right tool only when each step depends on the previous one, such as an
iteration or a recurrence, where there is nothing to vectorize.

In [ ]:
x = np.linspace(0, np.pi, 1000)

# loop version: accumulate one term at a time
s_loop = 0.0
for xi in x:
    s_loop += np.sin(xi)

# vectorized version: build all the sines, then sum
s_vec = np.sum(np.sin(x))

print("loop:", round(s_loop, 6), "  vectorized:", round(s_vec, 6), "  (identical)")

## 5. Solving a linear system

Lab A builds an interpolating polynomial by solving a linear system $Vc=y$ for the
coefficient vector $c$. Two rules. Use `np.linalg.solve(A, b)`, never `inv(A) @ b`;
solving is faster and more accurate than forming an inverse. And distinguish `@`, the
matrix product, from `*`, which is elementwise.

In [ ]:
# solve a 2x2 system  A c = b
Amat = np.array([[2.0, 1.0],
                 [1.0, 3.0]])
b = np.array([1.0, 2.0])
c = np.linalg.solve(Amat, b)
print("solution c   =", c)
print("residual A@c-b =", Amat @ c - b, "(should be ~0)")   # @ is matrix times vector

# The Vandermonde matrix has column k equal to (nodes)**k. np.vander builds it;
# increasing=True orders the columns as 1, x, x^2, ... to match p(x)=sum c[k] x**k.
xnodes = np.array([0.0, 1.0, 2.0])
V = np.vander(xnodes, len(xnodes), increasing=True)
print("Vandermonde V =\n", V)

## 6. Plotting

`matplotlib` draws curves. Open a figure, call `plot` once per curve, then label and
show. Use `semilogy` when the quantity ranges over many orders of magnitude, which is
exactly what an error-versus-degree curve does; on a linear axis such a curve would
be an unreadable spike.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.figsize"] = (8, 4.5)   # a comfortable default size

t = np.linspace(-1, 1, 200)
plt.figure()                                 # start a fresh figure
plt.plot(t, f(t), label="f")                 # first curve
plt.plot(t, 0.5 * np.ones_like(t), "k--", label="y = 0.5")   # dashed reference line
plt.xlabel("x"); plt.ylabel("y"); plt.legend()
plt.title("A simple plot"); plt.show()

# The same data on a log y-axis, the natural scale for errors.
n = np.arange(1, 10)
plt.figure(); plt.semilogy(n, 2.0**(-n), "r.-")   # "r.-" = red line with dots
plt.xlabel("n"); plt.ylabel("2^-n"); plt.title("Log scale on y"); plt.show()

## 7. The stub-and-self-check pattern

A lab gives you a function with its body missing (a stub, which raises
`NotImplementedError` until you write it) and a self-check cell that verifies your
version. `assert` raises an error if its condition is false and does nothing if true,
so a self-check that prints PASS means your code met the contract. `np.allclose`
compares arrays up to a small numerical tolerance, which is the right test for
floating-point results because exact equality almost never holds.

In [ ]:
def add_one(v):
    """Return v with 1 added to every entry."""
    return np.asarray(v) + 1        # in a lab, writing this line would be your job

# self-check: allclose tolerates tiny floating-point differences
assert np.allclose(add_one(np.array([0.0, 10.0])), np.array([1.0, 11.0]))
print("PASS  add_one")

## 8. Try it

Write `equispaced_nodes(a, b, n)` returning the `n+1` equally spaced points on
`[a, b]`, endpoints included. Attempt it yourself first; a solution follows so you
can compare. Remember the off-by-one from section 2: `n+1` points, `n` gaps.

In [ ]:
# one solution
def equispaced_nodes(a, b, n):
    """The n+1 equally spaced nodes on [a, b], endpoints included."""
    return np.linspace(a, b, n + 1)

# self-check
xn = equispaced_nodes(-1, 1, 4)
assert len(xn) == 5 and np.isclose(xn[0], -1) and np.isclose(xn[-1], 1)
print("PASS  equispaced_nodes:", xn)

## 9. Common errors, and how to read them

- `NotImplementedError`: you ran a stub before filling in its body.
- `NameError: name '...' is not defined`: you skipped, or have not yet run, the cell
  that defines it. Run the earlier cells, or Runtime > Restart and run all.
- `ValueError: operands could not be broadcast together with shapes ...`: two arrays
  in an elementwise operation have incompatible lengths. Print their `.shape` to see.
- A plot cell shows nothing: you forgot `plt.show()`, or you are drawing into a figure
  from a cell you did not re-run.
- Results look integer-ish or truncated: make sure your data is float. Writing `25.0`
  rather than `25`, or building arrays from floats, avoids surprises.

## 10. Using an AI assistant responsibly

You may use an AI tool to write lab code, and if your background is thin this is a
reasonable way to keep pace. Two habits keep it honest. Run the self-check cells:
they catch a wrong implementation no matter who wrote it. And keep the code separate
from the mathematics: the graded work is the derivations and the written answers,
which ask you to interpret the output the code produced. If you cannot explain why
the output looks the way it does, the assistant has not done your part of the work.

## 11. High precision, when a lab asks for it

Some labs compare ordinary `float64` (about 16 significant digits) against much
higher precision to decide whether an effect is genuine or merely rounding. The
`mpmath` library provides this: set the number of digits with `mp.mp.dps`, and wrap a
number in `mp.mpf` to carry it at that precision.

In [ ]:
import mpmath as mp
mp.mp.dps = 30                        # work to 30 significant digits
print("float64 :", 1/3)               # about 16 digits, the last few unreliable
print("mpmath  :", mp.mpf(1)/3)       # 30 correct digits

You now have what the labs assume. Open Lab A and begin.